<a href="https://colab.research.google.com/github/enyachang1119-debug/DS2002/blob/main/Copy_of_2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO

# Add a revenue column by multiplying quantity by price
df['revenue'] = df['qty'] * df['price']

# Calculate total revenue and total units
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print("Total revenue:", total_revenue)
print("Total units:", total_units)

Total revenue: 8520.0
Total units: 783


The 400 orders generated a total revenue of $8,520 and included 783 total units.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.


In [3]:
# TODO
# Calculate total revenue for each category
by_category = df.groupby('category')['revenue'].sum().reset_index()

# Calculate each category's percentage of total revenue
by_category['share_pct'] = by_category['revenue'] / total_revenue * 100

# Sort revenue from highest to lowest
by_category = by_category.sort_values('revenue', ascending=False)
print(by_category)

   category  revenue  share_pct
1      Food   4293.0  50.387324
2     Merch   1771.5  20.792254
0     Drink   1554.0  18.239437
3  RainGear    901.5  10.580986


Food generated the highest revenue at $4,293, which accounts for 50.39% of total revenue, followed by Merch, Drink, and RainGear.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO

# Calculate average order revenue and order count for each vendor
vendor_summary = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).reset_index()

# Sort by average order revenue from highest to lowest
vendor_summary = vendor_summary.sort_values('mean', ascending=False)
print(vendor_summary)

  vendor_id       mean  count
0      V-01  22.595745     94
3      V-18  21.750000    108
1      V-05  20.580645     93
2      V-10  20.314286    105


V-01 has the highest average order revenue at $22.60 across 94 orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO

# Calculate the share of total revenue from Merch
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = merch_revenue / total_revenue * 100

# Round the percentage to one decimal
merch_share = round(merch_share, 1)
print("Merch revenue share:", merch_share, "%")

Merch revenue share: 20.8 %


Merch accounts for 20.8% of the total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

# Save row count and total revenue before the merge
rows_before = len(df)
revenue_before = df['revenue'].sum()

# Left join vendor names to the orders
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

# Check row count and total revenue after the merge
rows_after = len(joined)
revenue_after = joined['revenue'].sum()
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Revenue before:", revenue_before)
print("Revenue after:", revenue_after)

# Find the unmatched vendor
unmatched_vendor = joined.loc[
    joined['vendor_name'].isna(),
    'vendor_id'
].unique()

print("Unmatched vendor:", unmatched_vendor)

Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0
Unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** The unmatched vendor is V-18. I kept its orders in the dataset and left the vendor name as missing because the correct vendor name is not provided in the lookup table. The merge kept all 400 rows, and the total revenue remained $8,520.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO

# Create a pivot table of revenue by vendor and category
revenue_pivot = pd.pivot_table(
    df,
    index='vendor_id',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(revenue_pivot)

category    Drink    Food   Merch  RainGear   Total
vendor_id                                          
V-01        171.0  1338.0   373.5     241.5  2124.0
V-05        298.5   882.0   489.0     244.5  1914.0
V-10        502.5  1054.5   400.5     175.5  2133.0
V-18        582.0  1018.5   508.5     240.0  2349.0
Total      1554.0  4293.0  1771.5     901.5  8520.0


The pivot table shows a total revenue of $8,520. V-18 generated the highest total revenue at $2,349, and Food was the highest-revenue category at $4,293.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.



```
# This is formatted as code
```

_a) For the next game, I would suggest the vendors to focus more on Food because it generated 50.4% of the total revenue, and Merch as well because it was the second-highest category in the revenue._

_b) I think Q6 is the least trustworthy answer because V-18 does not have a matching vendor name in the lookup table, which make it less informatie and should be interpreted carefully because we cannot identify the actual vendor name._
